<a href="https://colab.research.google.com/github/SyameimaruKoa/Minecraft-mod-AutoTranslation-tool/blob/main/run_colab_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minecraft Mod Auto Translator — Google Colab（LM Link）

ColabでLM Studio（llmster）を動かし、LM Link経由でPCと接続するのじゃ。認証情報は`dbv4-wd14-tagger`と共通のGoogleドライブフォルダーへ保存する。モデルはpfnet公式のGGUFファイルだけを使用し、既定は8B／Q4_K_Mじゃ。

## 1. サーバー設定

`dbv4-wd14-tagger`のColab版と同じ`USE_GOOGLE_DRIVE`と`DRIVE_FOLDER_NAME`を使用する。既定の保存先は `MyDrive/.colab-secret/` じゃ。

In [ ]:
#@title ⚙️ サーバー設定 (Configuration)
#@markdown dbv4-wd14-taggerと同じGoogleドライブ保存先設定じゃ。

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER_NAME = ".colab-secret" #@param {type:"string"}

CONFIG_DRIVE_FOLDER = DRIVE_FOLDER_NAME.strip("/ ") if DRIVE_FOLDER_NAME else ".colab-secret"
if (
    CONFIG_DRIVE_FOLDER in (".", "..")
    or "/" in CONFIG_DRIVE_FOLDER
    or "\\" in CONFIG_DRIVE_FOLDER
):
    raise ValueError("Googleドライブの保存先は単一フォルダー名を指定するのじゃ")
print(
    "[INFO] Googleドライブ保存: "
    + (f"有効 (MyDrive/{CONFIG_DRIVE_FOLDER})" if USE_GOOGLE_DRIVE else "無効")
)

## 2. LM Studioのインストール

先にインストールし、次のセルで認証情報を復元してからデーモンを起動する。

In [ ]:
#@title LM Studioのインストール
import os
import subprocess
from pathlib import Path

subprocess.run(["pkill", "-f", "llmster"], check=False)
subprocess.run(["pkill", "-f", "lmstudio"], check=False)
subprocess.run(["bash", "-c", "curl -fsSL https://lmstudio.ai/install.sh | bash"], check=True)
os.environ["PATH"] = str(Path.home() / ".lmstudio" / "bin") + os.pathsep + os.environ["PATH"]
subprocess.run(["lms", "bootstrap"], check=True)
subprocess.run(["sync"], check=True)
print("LM Studioのセットアップが完了したのじゃ。")

## 3. Googleドライブへのログイン状態保存

DBV4のTailscaleと同様、LM Studioのログイン状態をGoogleドライブの `MyDrive/.colab-secret/lmstudio/` 以下へ保存・自動復元する。パスフレーズの入力やColabシークレットは不要じゃ。保存対象は `credentials` と `hub` の認証状態のみで、内部デーモンキーやモデル本体は保存しない。

In [ ]:
#@title 認証情報の復元と保存
import os
import shutil
from pathlib import Path

from google.colab import drive

LM_HOME = Path.home() / ".lmstudio"
DRIVE_AUTH_DIR = None
AUTH_DIRS = ("credentials", "hub")

if USE_GOOGLE_DRIVE:
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    DRIVE_AUTH_DIR = Path("/content/drive/MyDrive") / CONFIG_DRIVE_FOLDER / "lmstudio"

def copy_login_files(source, destination):
    copied = 0
    for directory in AUTH_DIRS:
        root = source / directory
        if not root.is_dir() or root.is_symlink():
            continue
        for original in root.rglob("*"):
            if not original.is_file() or original.is_symlink():
                continue
            relative = original.relative_to(source)
            target = destination / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, target)
            os.chmod(target, 0o600)
            copied += 1
    return copied

def restore_login():
    if not USE_GOOGLE_DRIVE:
        return False
    restored = copy_login_files(DRIVE_AUTH_DIR, LM_HOME)
    if restored:
        print(f"[INFO] GoogleドライブからLM Link認証情報を復元: {restored}ファイル")
    else:
        print("[INFO] 保存済みのLM Link認証情報なし。初回ログインが必要じゃ。")
    return bool(restored)

def backup_login():
    if not USE_GOOGLE_DRIVE:
        return
    saved = copy_login_files(LM_HOME, DRIVE_AUTH_DIR)
    if not saved:
        print("[WARN] LM Linkは有効じゃが、認証ファイルが見つからずバックアップできなかったのじゃ。")
        print("[WARN] Colabのランタイムを作り直した際、再ログインが必要になる可能性があるぞ。")
        for directory_name in ("credentials", "hub", ".internal"):
            root = LM_HOME / directory_name
            count = sum(1 for entry in root.rglob("*") if entry.is_file() and not entry.is_symlink()) if root.is_dir() else 0
            print(f"[INFO] 保存候補 {directory_name}: {count}個の通常ファイル")
        for directory in (LM_HOME, Path.home() / ".config" / "LM Studio", Path.home() / ".cache" / "lm-studio"):
            if directory.is_dir():
                names = sorted(item.name for item in directory.iterdir() if item.is_dir() and not item.is_symlink())
                print(f"[INFO] 認証保存先の調査: {directory} / サブディレクトリー: {names}")
        return False
    print(f"[INFO] LM Link認証情報をGoogleドライブに保存: {saved}ファイル")
    return True

auth_restored = restore_login()

## 4. LM Linkのログインと接続

保存済み認証情報が有効なら再ログインせずに接続する。初回またはログイン期限切れなら `lms login` の認証URLをブラウザで開くのじゃ。ログイン後に認証情報を保存する。

Colabのフォームで `FORCE_RELOGIN` をオンにすると、一度ログアウトして疑似端末を使って認証URLを表示し、ブラウザ認証をやり直せる。認証ファイルの保存先は調査中じゃ。

In [ ]:
#@title LM Linkログインと接続
import errno
import html
import os
import pty
import re
import select
import subprocess
import time
from IPython.display import HTML, display

FORCE_RELOGIN = False #@param {type:"boolean"}

def login_using_pty():
    master, slave = pty.openpty()
    try:
        process = subprocess.Popen(
            ["lms", "login"], stdin=slave, stdout=slave, stderr=slave,
            start_new_session=True, close_fds=True
        )
    except Exception:
        os.close(master)
        os.close(slave)
        raise
    os.close(slave)
    started = time.monotonic()
    output = ""
    seen_urls = set()
    try:
        while True:
            ready, _, _ = select.select([master], [], [], 1.0)
            if ready:
                try:
                    data = os.read(master, 4096)
                except OSError as error:
                    if error.errno == errno.EIO:
                        break
                    raise
                if not data:
                    break
                chunk = data.decode("utf-8", errors="replace")
                print(chunk, end="", flush=True)
                output = (output + chunk)[-16384:]
                clean = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", output)
                links = re.findall(r"https://lmstudio\.ai/pairing(?:\?code=[A-Za-z0-9_-]+)?", clean)
                for url in links:
                    if url not in seen_urls:
                        display(HTML(
                            '<a href="' + html.escape(url, quote=True) +
                            '" target="_blank" rel="noopener">LM Studioの認証ページを開く</a>'
                        ))
                        seen_urls.add(url)
            if process.poll() is not None and not ready:
                break
            if not seen_urls and time.monotonic() - started > 90:
                raise TimeoutError("90秒経過してもLM Studioの認証URLが表示されないのじゃ。")
        code = process.wait(timeout=5)
        if code != 0:
            raise subprocess.CalledProcessError(code, ["lms", "login"])
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        os.close(master)

subprocess.run(["lms", "daemon", "up"], check=True)
time.sleep(3)
whoami = subprocess.run(
    ["lms", "whoami"], stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL, check=False, timeout=30
)
if FORCE_RELOGIN and whoami.returncode == 0:
    print("LM Studioから一度ログアウトし、再認証するのじゃ。", flush=True)
    subprocess.run(["lms", "logout"], check=True)
if whoami.returncode != 0 or FORCE_RELOGIN:
    print("LM Studioへのログインが必要じゃ。下に表示される認証URLで承認するのじゃ。", flush=True)
    login_using_pty()
    subprocess.run(["lms", "whoami"], check=True)
else:
    print("現在のLM Studioログインは有効じゃ。再認証にはFORCE_RELOGINを有効にするのじゃ。")
subprocess.run(["lms", "link", "enable"], check=True)
backup_login()
subprocess.run(["lms", "link", "status"], check=True)
print("LM Linkの接続処理が完了したのじゃ。")


## 5. PLaMo 3 Translateモデルのダウンロードとロード

既定は**8B／Q4_K_M**。0.8B・2B・8B・31Bからサイズを選び、pfnetが公式に配布しているGGUFの量子化形式だけ指定する。`lms get`の検索処理を使わず、PLaMo公式のGGUFファイルを直接取得するため、問題のコマンドは実行しない。ダウンロード中は進捗バー・転送量を表示し、中断時は可能なら途中から再開する。Colabランタイムを破棄するとモデルファイルは消えるが、Googleドライブの保存済み認証情報は残る。31BはGPUメモリ不足でロードできない場合があるのじゃ。

In [ ]:
#@title PLaMoモデルの選択と読み込み
import os
import subprocess
from pathlib import Path

plamo_size = "8b" # @param ["0.8b", "2b", "8b", "31b"]
quantization = "Q4_K_M" # @param ["Q4_K_M", "Q3_K_M", "Q2_K", "Q4_0"]

OFFICIAL_GGUF_FORMATS = {
    "0.8b": ("Q4_K_M", "Q4_0", "Q3_K_M", "Q2_K"),
    "2b": ("Q4_K_M", "Q4_0", "Q3_K_M", "Q2_K"),
    "8b": ("Q4_K_M", "Q3_K_M", "Q2_K"),
    "31b": ("Q4_K_M", "Q3_K_M", "Q2_K"),
}
if plamo_size not in OFFICIAL_GGUF_FORMATS:
    raise ValueError("未対応のPLaMoサイズじゃ")
if quantization not in OFFICIAL_GGUF_FORMATS[plamo_size]:
    raise ValueError(f"pfnet公式で配布されていない組み合わせじゃ: {plamo_size}/{quantization}")

repo_name = f"plamo-3-translate-{plamo_size}-GGUF"
filename = f"plamo-3-translate-{plamo_size}-{quantization}.gguf"
model_dir = Path.home() / ".lmstudio" / "models" / "pfnet" / repo_name
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / filename
partial_path = model_dir / (filename + ".part")
model_id = f"plamo-3-translate-{plamo_size}-colab"
url = f"https://huggingface.co/pfnet/{repo_name}/resolve/main/{filename}?download=true"

if not model_path.exists() or model_path.stat().st_size < 100_000_000:
    print(f"公式GGUFを取得: pfnet/{repo_name}/{filename}", flush=True)
    import re
    import time
    import requests
    from tqdm.auto import tqdm

    def download_with_progress(source_url, partial, complete):
        for attempt in range(1, 5):
            offset = partial.stat().st_size if partial.exists() else 0
            headers = {"Range": f"bytes={offset}-"} if offset else {}
            try:
                with requests.get(source_url, headers=headers, stream=True, timeout=(30, 120)) as response:
                    if response.status_code == 416 and offset:
                        match = re.fullmatch(r"bytes \*/(\d+)", response.headers.get("Content-Range", ""))
                        if match and int(match.group(1)) == offset:
                            os.replace(partial, complete)
                            return
                    response.raise_for_status()
                    if response.status_code == 206:
                        match = re.fullmatch(
                            r"bytes (\d+)-(\d+)/(\d+|\*)",
                            response.headers.get("Content-Range", "")
                        )
                        if not match or int(match.group(1)) != offset:
                            raise RuntimeError("ダウンロードの再開位置が一致しないのじゃ")
                        total = int(match.group(3)) if match.group(3) != "*" else None
                        mode = "ab"
                    elif response.status_code == 200:
                        if offset:
                            print("サーバーが再開を受け付けなかったため、先頭から再取得するのじゃ。", flush=True)
                        offset = 0
                        total = int(response.headers["Content-Length"]) if response.headers.get("Content-Length") else None
                        mode = "wb"
                    else:
                        raise RuntimeError(f"予期しないダウンロード応答: HTTP {response.status_code}")
                    print(f"ダウンロード開始: {offset / 1048576:.1f} MiB取得済み", flush=True)
                    with partial.open(mode) as output, tqdm(
                        total=total, initial=offset, unit="B", unit_scale=True,
                        desc="PLaMo GGUF", mininterval=0.5, leave=True
                    ) as progress:
                        for chunk in response.iter_content(chunk_size=1048576):
                            if chunk:
                                output.write(chunk)
                                progress.update(len(chunk))
                    current = partial.stat().st_size
                    if total is not None and current != total:
                        raise RuntimeError(f"サイズ不一致: {current} / {total}バイト")
                    if current < 100_000_000:
                        raise RuntimeError("GGUFのダウンロード結果が小さすぎるのじゃ")
                    os.replace(partial, complete)
                    print("GGUFのダウンロードが完了したのじゃ。", flush=True)
                    return
            except (requests.RequestException, OSError) as error:
                if attempt == 4:
                    raise
                print(f"通信が中断したのじゃ（{attempt}/4）: {error}。途中から再試行するぞ。", flush=True)
                time.sleep(min(attempt * 2, 8))
        raise RuntimeError("GGUFのダウンロードに失敗したのじゃ")

    download_with_progress(url, partial_path, model_path)
else:
    print(f"ダウンロード済みGGUFを再利用: {filename}", flush=True)

subprocess.run(["lms", "unload", "--all"], check=True)
subprocess.run(
    ["lms", "load", str(model_path), "--gpu", "max", "--identifier", model_id],
    check=True
)
subprocess.run(["lms", "server", "start", "--port", "1234"], check=True)
subprocess.run(["lms", "ps"], check=True)
print(f"ロード完了: {model_id}（LM Linkから利用可能）", flush=True)